# Exercises on Tree-Based Models

In [57]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 1. Generate a complex synthetic dataset
# n_informative: Features that actually correlate with 'y'
# n_redundant: Linear combinations of informative features (tests model robustness)
# n_clusters_per_class: Increases complexity/overlap (leads to overfitting)
X, y = make_classification(
    n_samples=500,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    n_repeated=0,
    n_clusters_per_class=2,
    flip_y=0.1, # Adds 10% random noise to the target to encourage overfitting
    random_state=42
)

# 2. Convert to DataFrame
feature_names = [f'feature_{i+1}' for i in range(5)]
df_simulated = pd.DataFrame(X, columns=feature_names)
df_simulated['y'] = y

# 3. Create a split for testing underfitting/overfitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Dataset generated with {df_simulated.shape[0]} rows and 5 features.")
print(df_simulated.head())

Dataset generated with 500 rows and 5 features.
   feature_1  feature_2  feature_3  feature_4  feature_5  y
0   2.403488  -0.194466   0.667461  -0.428694   1.945634  0
1  -1.670430  -0.700467   0.831349  -0.591764  -0.824777  0
2   1.342949   0.594975   0.236582   1.738537   0.470102  1
3   0.168406   0.090667  -1.034373  -0.809959   0.195843  0
4  -2.031498  -0.873334  -0.693187   0.043145  -1.100043  0


## 1. Decision trees

a) Compute the **Gini coefficient** for the data set The Gini Impurity for a set with classes $i \in \{1, 2, ..., C\}$ is calculated as:
$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

b) Create a **decision tree** of max_depth=1 for the data set. Identify the best split point (stump) among the features $x_1$ and $x_2$ that minimizes the weighted average Gini impurity.

c) **Experiment** with different values of max_depth and min_samples_split to control the complexity of the tree. Evaluate the performance of the tree on the train and test set.

In [58]:
# a

n = len(df_simulated)

class_counts = df_simulated['y'].value_counts()

sum_squared_probabilities = 0

for label, count in class_counts.items():
    p_i = count / n
    sum_squared_probabilities += p_i**2
    print(f"Class {label}: count={count}, p={p_i}, p^2={p_i**2:.3f}")

gini_impurity = 1 - sum_squared_probabilities

print(f"\nFinal Gini Impurity: {gini_impurity:.3f}")

Class 0: count=255, p=0.51, p^2=0.260
Class 1: count=245, p=0.49, p^2=0.240

Final Gini Impurity: 0.500


In [59]:
# b

# create a decision tree stump (max_depth=1)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
# separate features and target variable
X = df_simulated.drop(columns=["y"])
y = df_simulated["y"]
# split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# train a decision tree classifier with max_depth=1
tree_stump = DecisionTreeClassifier(max_depth=1, random_state=42)
tree_stump.fit(X_train, y_train)
# evaluate the stump on the test set
y_pred_stump = tree_stump.predict(X_test)
accuracy_stump = accuracy_score(y_test, y_pred_stump)
print(f"Test set accuracy of decision tree stump: {accuracy_stump:.2f}")

# training accuracy
y_train_pred_stump = tree_stump.predict(X_train)
train_accuracy_stump = accuracy_score(y_train, y_train_pred_stump)
print(f"Train set accuracy of decision tree stump: {train_accuracy_stump:.2f}")

Test set accuracy of decision tree stump: 0.81
Train set accuracy of decision tree stump: 0.84


In [60]:
# c

# for loop to experiment with different hyperparameters
for max_depth in [1, 2, 3, None]:
    for min_samples_split in [2, 5, 10]:
        tree_clf = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)
        tree_clf.fit(X_train, y_train)
        y_pred = tree_clf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        y_train_pred = tree_clf.predict(X_train)
        train_accuracy = accuracy_score(y_train, y_train_pred)
        print(f"max_depth={max_depth}, min_samples_split={min_samples_split} => Test set accuracy: {accuracy:.2f}, Train set accuracy: {train_accuracy:.2f}")

max_depth=1, min_samples_split=2 => Test set accuracy: 0.81, Train set accuracy: 0.84
max_depth=1, min_samples_split=5 => Test set accuracy: 0.81, Train set accuracy: 0.84
max_depth=1, min_samples_split=10 => Test set accuracy: 0.81, Train set accuracy: 0.84
max_depth=2, min_samples_split=2 => Test set accuracy: 0.89, Train set accuracy: 0.88
max_depth=2, min_samples_split=5 => Test set accuracy: 0.89, Train set accuracy: 0.88
max_depth=2, min_samples_split=10 => Test set accuracy: 0.89, Train set accuracy: 0.88
max_depth=3, min_samples_split=2 => Test set accuracy: 0.87, Train set accuracy: 0.89
max_depth=3, min_samples_split=5 => Test set accuracy: 0.87, Train set accuracy: 0.89
max_depth=3, min_samples_split=10 => Test set accuracy: 0.87, Train set accuracy: 0.89
max_depth=None, min_samples_split=2 => Test set accuracy: 0.86, Train set accuracy: 1.00
max_depth=None, min_samples_split=5 => Test set accuracy: 0.84, Train set accuracy: 0.97
max_depth=None, min_samples_split=10 => Test 

## 2. Random Forests

a) Train a **random forest** classifier on the data set. Use n_estimators=100 and max_depth=4. Evaluate the performance of the random forest on a test set.

b) Experiment with different values of **hyperparameters** (ex. n_estimators, n_features, max_depth) to control the complexity of the random forest. Evaluate the performance of the random forest. 

c) Compare the performance of the decision tree and random forest models. Which one **performs better** on the test set? Why do you think that is the case?



In [61]:
# a

from sklearn.ensemble import RandomForestClassifier
n_features = X_train.shape[1]
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42, max_features=int(np.sqrt(n_features)))
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Test set accuracy of random forest: {accuracy_rf:.2f}")
# training accuracy
y_train_pred_rf = rf_clf.predict(X_train)
train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
print(f"Train set accuracy of random forest: {train_accuracy_rf:.2f}")

Test set accuracy of random forest: 0.86
Train set accuracy of random forest: 0.91


In [62]:
# b

# for loop to experiment with different hyperparameters
for n_estimators in [50, 100, 200]:
    for max_depth in [2, 4, None]:
        rf_clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, max_features=int(np.sqrt(n_features)))
        rf_clf.fit(X_train, y_train)
        y_pred_rf = rf_clf.predict(X_test)
        accuracy_rf = accuracy_score(y_test, y_pred_rf)
        y_train_rf = rf_clf.predict(X_train)
        train_accuracy_rf = accuracy_score(y_train, y_train_rf)
        print(f"n_estimators={n_estimators}, max_depth={max_depth} => Test set accuracy: {accuracy_rf:.2f}, Train set accuracy: {train_accuracy_rf:.2f}")

n_estimators=50, max_depth=2 => Test set accuracy: 0.86, Train set accuracy: 0.86
n_estimators=50, max_depth=4 => Test set accuracy: 0.87, Train set accuracy: 0.91
n_estimators=50, max_depth=None => Test set accuracy: 0.87, Train set accuracy: 1.00
n_estimators=100, max_depth=2 => Test set accuracy: 0.86, Train set accuracy: 0.88
n_estimators=100, max_depth=4 => Test set accuracy: 0.86, Train set accuracy: 0.91
n_estimators=100, max_depth=None => Test set accuracy: 0.87, Train set accuracy: 1.00
n_estimators=200, max_depth=2 => Test set accuracy: 0.87, Train set accuracy: 0.89
n_estimators=200, max_depth=4 => Test set accuracy: 0.86, Train set accuracy: 0.91
n_estimators=200, max_depth=None => Test set accuracy: 0.87, Train set accuracy: 1.00


## 3. Gradient-Boosted Trees

a) Train a **gradient-boosted tree** classifier on the data set. Use n_estimators=100 and learning_rate=0.1. Evaluate the performance of the gradient-boosted tree on a test set.

b) Experiment with different values of **hyperparameters** (ex. n_estimators, learning_rate, max_depth) to control the complexity of the gradient-boosted tree. Evaluate the performance of the gradient-boosted tree.

c) Idendify when your model is **overfitting or underfitting**. What hyperparameters would you adjust to address overfitting or underfitting in your model?

In [63]:
# a

from sklearn.ensemble import GradientBoostingClassifier
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)
accuracy_gb = accuracy_score(y_test, y_pred_gb)
print(f"Test set accuracy of gradient-boosted tree: {accuracy_gb:.2f}")
# training accuracy
y_train_pred_gb = gb_clf.predict(X_train)
train_accuracy_gb = accuracy_score(y_train, y_train_pred_gb)
print(f"Train set accuracy of gradient-boosted tree: {train_accuracy_gb:.2f}")

Test set accuracy of gradient-boosted tree: 0.86
Train set accuracy of gradient-boosted tree: 0.98


In [64]:
# b

# for loop to experiment with different hyperparameters
for n_estimators in [50, 100, 200]:
    for learning_rate in [0.01, 0.1, 0.2]:
        gb_clf = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, random_state=42)
        gb_clf.fit(X_train, y_train)
        y_pred_gb = gb_clf.predict(X_test)
        accuracy_gb = accuracy_score(y_test, y_pred_gb)
        y_train_gb = gb_clf.predict(X_train)
        train_accuracy_gb = accuracy_score(y_train, y_train_gb)
        print(f"n_estimators={n_estimators}, learning_rate={learning_rate} => Test set accuracy: {accuracy_gb:.2f}, train set accuracy: {train_accuracy_gb:.2f}")


n_estimators=50, learning_rate=0.01 => Test set accuracy: 0.86, train set accuracy: 0.90
n_estimators=50, learning_rate=0.1 => Test set accuracy: 0.85, train set accuracy: 0.94
n_estimators=50, learning_rate=0.2 => Test set accuracy: 0.86, train set accuracy: 0.99
n_estimators=100, learning_rate=0.01 => Test set accuracy: 0.86, train set accuracy: 0.90
n_estimators=100, learning_rate=0.1 => Test set accuracy: 0.86, train set accuracy: 0.98
n_estimators=100, learning_rate=0.2 => Test set accuracy: 0.86, train set accuracy: 1.00
n_estimators=200, learning_rate=0.01 => Test set accuracy: 0.87, train set accuracy: 0.91
n_estimators=200, learning_rate=0.1 => Test set accuracy: 0.86, train set accuracy: 1.00
n_estimators=200, learning_rate=0.2 => Test set accuracy: 0.86, train set accuracy: 1.00
